# IBM Nighthawk RAW noisy baseline generator (qisk)

Original qisk script `vqe_noisy_baseline.py`. Runs the closed-loop noisy VQE on the FakeNighthawk backend (no mitigation) and writes `vqe_noisy_baseline_raw.pkl`, which is stored here as `results/hardware/ibm_nighthawk_baseline_raw.pkl` (consumed by qpu_phenomenological_relations.ipynb for Fig 10). Requires the qiskit + qiskit-aer + qiskit-ibm-runtime environment (NOT needed to render figures).

In [ ]:
"""
VQE noisy baseline: HVA ansatz (RXX/RYY/RZZ) on FakeNighthawk, no mitigation.

Purpose: establish the raw noisy performance as a function of circuit depth,
to motivate the use of error mitigation. Results are saved and plotted:

  (a) E(N_it) - E_∞ convergence trajectories per layer
  (b) E_∞ vs N_g (and N_layers on dual x-axis), with noiseless overlay and E_gs

Outputs:
  vqe_noisy_baseline_raw.pkl
  Figs/noisy_baseline_convergence.pdf
  Figs/noisy_baseline_Einf_vs_Ng.pdf
"""

import warnings
warnings.filterwarnings("ignore", message=".*no effect in local testing mode.*")
warnings.filterwarnings("ignore", message=".*Properties of fake.*")

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from itertools import combinations
from scipy.optimize import minimize, curve_fit
from joblib import Parallel, delayed

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import RXXGate, RYYGate, RZZGate
from qiskit.transpiler import generate_preset_pass_manager, Layout
from qiskit.primitives import StatevectorEstimator

from qiskit_ibm_runtime.fake_provider import (
    FakeNighthawk, FakeKyiv, FakeTorino, FakeSherbrooke
)
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime import EstimatorV2 as Estimator

os.makedirs("Figs", exist_ok=True)
plt.rcParams.update({"font.size": 18})
plt.rcParams["text.usetex"] = False


# ── Experiment config ─────────────────────────────────────────────────────────

LAYER_DEPTHS = [1, 2, 3, 4, 5, 6]
NUM_SEEDS    = 20


# ── Hamiltonian ───────────────────────────────────────────────────────────────

def heisenberg_hamiltonian(nqbt):
    paulis, coeffs = [], []
    for i in range(nqbt - 1):
        paulis += [
            "I"*i + "XX" + "I"*(nqbt-i-2),
            "I"*i + "YY" + "I"*(nqbt-i-2),
            "I"*i + "ZZ" + "I"*(nqbt-i-2),
        ]
        coeffs += [1.0, 1.0, 1.0]
    return SparsePauliOp(paulis, coeffs)


# ── Variational circuit (HVA, native Pauli-exponential form) ──────────────────

def variational_circuit(nqbt, ct):
    # 6 params per layer: 3 shared across all odd bonds, 3 shared across all even bonds.
    # This is the correct HVA structure — translational symmetry of the Heisenberg chain
    # means all bonds in the same parity sublayer are governed by the same angles.
    n_params = 6 * ct
    params   = ParameterVector("θ", n_params)
    qc       = QuantumCircuit(nqbt)
    for q in range(nqbt):
        qc.x(q)
    for q in range(0, nqbt, 2):
        qc.h(q)
        if q + 1 < nqbt:
            qc.cx(q, q + 1)
    for it in range(ct):
        p = 6 * it
        # Odd bonds: (1,2), (3,4), ... — all share params p, p+1, p+2
        for q in range(1, nqbt - 1, 2):
            qc.append(RXXGate(params[p + 0]), [q, q + 1])
            qc.append(RYYGate(params[p + 1]), [q, q + 1])
            qc.append(RZZGate(params[p + 2]), [q, q + 1])
        # Even bonds: (0,1), (2,3), ... — all share params p+3, p+4, p+5
        for q in range(0, nqbt - 1, 2):
            qc.append(RXXGate(params[p + 3]), [q, q + 1])
            qc.append(RYYGate(params[p + 4]), [q, q + 1])
            qc.append(RZZGate(params[p + 5]), [q, q + 1])
    return qc


# ── Gate counting (manuscript convention) ─────────────────────────────────────

def count_gates(circuit: QuantumCircuit) -> int:
    ng = 0
    for inst in circuit.data:
        ng += 2 if inst.operation.num_qubits >= 2 else 1
    return ng


# ── Backend selection: pick QPU with lowest mean native 2Q error ──────────────

num_qubits = 5

_CANDIDATE_BACKENDS = [FakeNighthawk, FakeKyiv, FakeTorino, FakeSherbrooke]

def _native_2q_gate(backend_instance):
    """Return the primary native 2Q gate name for this backend."""
    gates_2q = {
        g["gate"]
        for g in backend_instance.properties().to_dict().get("gates", [])
        if len(g["qubits"]) == 2
    }
    for preferred in ("cx", "ecr", "cz", "rzz"):
        if preferred in gates_2q:
            return preferred
    return next(iter(gates_2q), "cx")

def _mean_2q_error(backend_instance):
    native = _native_2q_gate(backend_instance)
    errors = [
        g["parameters"][0]["value"]
        for g in backend_instance.properties().to_dict().get("gates", [])
        if g["gate"] == native
    ]
    return sum(errors) / len(errors) if errors else float("inf")

def select_best_qubits(backend_instance, nq):
    native = _native_2q_gate(backend_instance)
    gate_errors = {
        tuple(gate["qubits"]): gate["parameters"][0]["value"]
        for gate in backend_instance.properties().to_dict().get("gates", [])
        if gate["gate"] == native
    }
    best, best_err = None, float("inf")
    for qs in combinations(range(backend_instance.num_qubits), nq):
        err = sum(gate_errors.get((qs[i], qs[i+1]), 1.0) for i in range(nq - 1))
        if err < best_err:
            best_err, best = err, qs
    return list(best)

print("=" * 60)
print("QPU selection by mean native 2Q error rate")
print("=" * 60)
_backend_scores = []
for cls in _CANDIDATE_BACKENDS:
    b = cls()
    native   = _native_2q_gate(b)
    mean_err = _mean_2q_error(b)
    best_qs  = select_best_qubits(b, num_qubits)
    gate_errs = {
        tuple(g["qubits"]): g["parameters"][0]["value"]
        for g in b.properties().to_dict().get("gates", [])
        if g["gate"] == native
    }
    best_chain_err = sum(
        gate_errs.get((best_qs[i], best_qs[i+1]), 1.0) for i in range(num_qubits - 1)
    )
    _backend_scores.append((mean_err, cls, b, best_qs, best_chain_err))
    print(f"  {b.name:<22}  native={native:<3}  mean 2Q = {mean_err:.5f}  "
          f"best-chain = {best_chain_err:.5f}  qubits = {best_qs}")

_backend_scores.sort(key=lambda x: x[0])
_, _best_cls, backend, selected_qubits, _ = _backend_scores[0]
print(f"\n  Selected backend        : {backend.name}  (lowest mean 2Q error)")
print("=" * 60)

noise_model = NoiseModel.from_backend(backend)
print(f"  Basis gates with errors : {noise_model.basis_gates}")
print(f"  Qubits with errors      : {sorted(noise_model.noise_qubits)[:10]} ...")
print(f"  Selected qubits         : {selected_qubits}")
print("=" * 60)

noisy_sim     = AerSimulator(method="statevector").from_backend(backend)
noiseless_sim = AerSimulator(method="statevector")

qr = QuantumRegister(num_qubits, "q")
initial_layout = Layout({qr[i]: q for i, q in enumerate(selected_qubits)})
pm = generate_preset_pass_manager(
    backend=noisy_sim,
    optimization_level=3,
    layout_method="trivial",
    initial_layout=initial_layout,
    qubits_initially_zero=True,
)


# ── Exact ground state energy ─────────────────────────────────────────────────

obs_exact    = heisenberg_hamiltonian(num_qubits)
ham_matrix   = np.array(obs_exact)
exact_energy = min(np.linalg.eig(ham_matrix)[0]).real
print(f"Exact ground state energy: E_gs = {exact_energy:.6f}\n")


# ── Cost functions ────────────────────────────────────────────────────────────

def cost_func_noisy(params, ansatz, hamiltonian, estimator, cost_history):
    energy = float(np.squeeze(
        estimator.run([(ansatz.assign_parameters(params), hamiltonian)])
        .result()[0].data.evs
    ))
    cost_history.append(energy)
    return energy

def cost_func_noiseless(params, ansatz, hamiltonian, estimator, cost_history):
    energy = float(np.squeeze(
        estimator.run([(ansatz.assign_parameters(params), hamiltonian)])
        .result()[0].data.evs
    ))
    cost_history.append(energy)
    return energy


# ── Seed runners ──────────────────────────────────────────────────────────────

def _minimize(cost_fn, params, ansatz, observable, estimator, cost_history):
    bounds = [(0, 2 * np.pi)] * len(params)
    res = minimize(
        cost_fn, x0=params,
        args=(ansatz, observable, estimator, cost_history),
        method="cobyla", bounds=bounds,
        options={"maxiter": 10000, "tol": 1e-8, "disp": False},
    )
    return cost_history, res.fun


def run_seed_noiseless(seed_idx, ansatz_logical, observable_logical):
    est = StatevectorEstimator()
    np.random.seed(seed_idx)
    params = 2 * np.pi * np.random.random(ansatz_logical.num_parameters)
    return _minimize(cost_func_noiseless, params, ansatz_logical, observable_logical, est, [])


def run_seed_noisy(seed_idx, ansatz_ibm, observable_ibm):
    _backend = _best_cls()
    sim = AerSimulator(method="statevector").from_backend(_backend)
    est = Estimator(sim)
    np.random.seed(seed_idx)
    params = 2 * np.pi * np.random.random(ansatz_ibm.num_parameters)
    return _minimize(cost_func_noisy, params, ansatz_ibm, observable_ibm, est, [])


# ── Main loop ─────────────────────────────────────────────────────────────────

noiseless_results = {}
noisy_results     = {}
Ng_by_layer       = {}

for layer_depth in LAYER_DEPTHS:
    ansatz         = variational_circuit(num_qubits, layer_depth)
    ansatz_ibm     = pm.run(ansatz)
    observable_ibm = heisenberg_hamiltonian(num_qubits).apply_layout(ansatz_ibm.layout)
    Ng             = count_gates(ansatz_ibm)
    Ng_by_layer[layer_depth] = Ng

    print(f"\n{'='*60}")
    print(f"Layer depth {layer_depth}  |  Ng = {Ng}  |  params = {ansatz_ibm.num_parameters}")
    print("="*60)

    print("  [noiseless]")
    observable_logical = heisenberg_hamiltonian(num_qubits)
    nl_res = Parallel(n_jobs=-1)(
        delayed(run_seed_noiseless)(s, ansatz, observable_logical)
        for s in range(NUM_SEEDS)
    )
    nl_hists, nl_finals = zip(*nl_res)
    noiseless_results[layer_depth] = {
        "cost_histories": nl_hists, "final_energies": nl_finals, "Ng": Ng
    }
    print(f"  Noiseless avg : {np.mean(nl_finals):.4f} ± {np.std(nl_finals):.4f}")

    print("  [noisy, no mitigation]")
    ny_res = Parallel(n_jobs=-1)(
        delayed(run_seed_noisy)(s, ansatz_ibm, observable_ibm)
        for s in range(NUM_SEEDS)
    )
    ny_hists, ny_finals = zip(*ny_res)
    noisy_results[layer_depth] = {
        "cost_histories": ny_hists, "final_energies": ny_finals, "Ng": Ng
    }
    print(f"  Noisy avg     : {np.mean(ny_finals):.4f} ± {np.std(ny_finals):.4f}")


# ── Save raw data ─────────────────────────────────────────────────────────────

with open("vqe_noisy_baseline_raw.pkl", "wb") as f:
    pickle.dump({
        "noiseless_results": noiseless_results,
        "noisy_results":     noisy_results,
        "Ng_by_layer":       Ng_by_layer,
        "exact_energy":      exact_energy,
        "layer_depths":      LAYER_DEPTHS,
        "num_seeds":         NUM_SEEDS,
    }, f)
print("\nSaved vqe_noisy_baseline_raw.pkl")


# ── Plotting setup ────────────────────────────────────────────────────────────

layer_list = sorted(noiseless_results.keys())
Ng_arr     = np.array([noiseless_results[ld]["Ng"] for ld in layer_list], dtype=float)
gate_ctt   = Ng_arr.astype(int)
n_lay      = len(layer_list)
colors     = plt.cm.tab10(np.linspace(0, 0.9, n_lay))

def _padded_mean_std(histories):
    max_l  = max(len(h) for h in histories)
    padded = np.array([list(h) + [h[-1]] * (max_l - len(h)) for h in histories])
    return np.mean(padded, axis=0), np.std(padded, axis=0)

val_nl_m, val_nl_s = [], []
val_ny_m, val_ny_s = [], []
nl_mean_traj, nl_std_traj = {}, {}
ny_mean_traj, ny_std_traj = {}, {}

for ld in layer_list:
    m, s = _padded_mean_std(noiseless_results[ld]["cost_histories"])
    nl_mean_traj[ld], nl_std_traj[ld] = m, s
    val_nl_m.append(np.mean(noiseless_results[ld]["final_energies"]))
    val_nl_s.append(np.std(noiseless_results[ld]["final_energies"]))

    m, s = _padded_mean_std(noisy_results[ld]["cost_histories"])
    ny_mean_traj[ld], ny_std_traj[ld] = m, s
    val_ny_m.append(np.mean(noisy_results[ld]["final_energies"]))
    val_ny_s.append(np.std(noisy_results[ld]["final_energies"]))

val_nl_m = np.array(val_nl_m)
val_nl_s = np.array(val_nl_s)
val_ny_m = np.array(val_ny_m)
val_ny_s = np.array(val_ny_s)


def exp_model(t, a, b):
    return a * np.exp(-b * t)


# ── Plot (a): convergence trajectories — noiseless | noisy side by side ───────

fit_ran  = 25
xlim_max = fit_ran * n_lay
xx       = np.arange(0, 25000, dtype=int)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, (mean_traj, std_traj, val_m, panel_label) in zip(
    axes,
    [
        (nl_mean_traj, nl_std_traj, val_nl_m, "a"),
        (ny_mean_traj, ny_std_traj, val_ny_m, "b"),
    ],
):
    for i, (ld, color) in enumerate(zip(layer_list, colors)):
        fit_range = (i + 1) * fit_ran
        yy = mean_traj[ld] - val_m[i]
        sd = std_traj[ld]
        xp = xx[:fit_range]
        yp = yy[:fit_range]
        ep = sd[:fit_range]

        ax.errorbar(xp, yp, yerr=ep, fmt="-", color=color,
                    label=r"$%d~(N_\text{g}=%d)$" % (ld, gate_ctt[i]))

        valid = yp > 1e-8
        if valid.sum() >= 5:
            try:
                params_fit, _ = curve_fit(
                    exp_model, xp[valid], yp[valid],
                    sigma=ep[valid] + 1e-12, absolute_sigma=True,
                    p0=[max(yp[valid]), 0.05],
                    bounds=([0, 1e-8], [np.inf, 10]), maxfev=10000,
                )
                x_fit = np.linspace(0, fit_range, 200)
                ax.plot(x_fit, exp_model(x_fit, *params_fit),
                        "--k", lw=1.2, alpha=0.6)
            except RuntimeError:
                pass

    ax.set_xlabel(r"$\text{Number of Iterations,}~N_\text{it}$", fontsize=18)
    ax.set_ylabel(r"$E(N_\text{it}) - E_{\infty}$", fontsize=18)
    ax.set_xlim(0, xlim_max)
    ax.set_ylim(bottom=0)
    ax.legend(ncol=2, loc="upper right",
               title=r"$N_\text{layers}$", title_fontsize=16,
               fontsize=15, labelspacing=0.3, handletextpad=0.3)
    ax.text(0.02, 0.97, f"({panel_label})", transform=ax.transAxes,
            fontsize=18, verticalalignment="top")
    ax.grid(True, alpha=0.3)

axes[0].set_title("Noiseless", fontsize=18, pad=8)
axes[1].set_title("Noisy (no mitigation)", fontsize=18, pad=8)

plt.tight_layout()
plt.savefig("Figs/noisy_baseline_convergence.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Saved Figs/noisy_baseline_convergence.pdf")


# ── Plot (b): E_∞ vs N_g, noiseless + noisy, dual x-axis ─────────────────────

fig, ax = plt.subplots(figsize=(8, 6))

ax.errorbar(gate_ctt, val_nl_m, yerr=val_nl_s,
            fmt="d-", markersize=10, color="black", linestyle="-",
            label=r"$\epsilon = 0~\text{(noiseless)}$")
ax.errorbar(gate_ctt, val_ny_m, yerr=val_ny_s,
            fmt="o", markersize=10, color="#D62728", linestyle="-.",
            barsabove=True,
            label=r"$\text{Noisy (no mitigation)}$")
ax.axhline(exact_energy, color="m", linestyle="--", lw=1.5,
           label=r"$E_\text{gs}$")

ax.set_xlabel(r"$\text{Number of gates, }N_\text{g}$", fontsize=18)
ax.set_ylabel(r"$\text{Converged VQE energy,}~E_\infty$", fontsize=18)
ax.set_xticks(gate_ctt)
ax.set_xticklabels(gate_ctt, rotation=0, ha="right")

axx = ax.secondary_xaxis("top")
axx.set_xticks(gate_ctt)
axx.set_xticklabels(np.arange(1, n_lay + 1, dtype=int))
axx.set_xlabel(r"$N_\text{layers}$", fontsize=18)

ax.legend(ncol=1, bbox_to_anchor=(1, 1.35), ncols= 3, fontsize=16,
           labelspacing=0.4, handletextpad=0.4)
ax.text(0.02, 0.97, "(c)", transform=ax.transAxes,
        fontsize=18, verticalalignment="top")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("Figs/noisy_baseline_Einf_vs_Ng.pdf", bbox_inches="tight", dpi=300)
plt.show()
print("Saved Figs/noisy_baseline_Einf_vs_Ng.pdf")

#---------------------new plots ---------------------------------

with open("vqe_noisy_baseline_raw.pkl", "rb") as f:
    data = pickle.load(f)

noiseless_results = data["noiseless_results"]
noisy_results     = data["noisy_results"]
exact_energy      = data["exact_energy"]
layer_list        = sorted(noiseless_results.keys())
n_lay             = len(layer_list)
colors            = plt.cm.tab10(np.linspace(0, 0.9, n_lay))

def _padded_mean_std(histories):
    max_l  = max(len(h) for h in histories)
    padded = np.array([list(h) + [h[-1]] * (max_l - len(h)) for h in histories])
    return np.mean(padded, axis=0), np.std(padded, axis=0)

def exp_model(t, a, b):
    return a * np.exp(-b * t)

fit_ran = 25

for results, label, fname in [
    (noiseless_results, "Noiseless",           "Figs/noisy_baseline_convergence_noiseless_log.pdf"),
    (noisy_results,     "Noisy (no mitigation)", "Figs/noisy_baseline_convergence_noisy_log.pdf"),
]:
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (ld, color) in enumerate(zip(layer_list, colors)):
        Ng        = results[ld]["Ng"]
        val_m     = np.mean(results[ld]["final_energies"])
        m, s      = _padded_mean_std(results[ld]["cost_histories"])
        fit_range = (i + 1) * fit_ran
        xx        = np.arange(fit_range)
        yy        = m[:fit_range] - val_m
        sd        = s[:fit_range]

        # keep only positive values for log plot
        pos = yy > 0
        ax.semilogy(xx[pos], yy[pos], color=color,
                    label=r"$%d~(N_\mathrm{g}=%d)$" % (ld, Ng))

        if pos.sum() >= 5:
            try:
                pfit, _ = curve_fit(
                    exp_model, xx[pos], yy[pos],
                    sigma=sd[pos] + 1e-12, absolute_sigma=True,
                    p0=[max(yy[pos]), 0.05],
                    bounds=([0, 1e-8], [np.inf, 10]), maxfev=10000,
                )
                x_fit = np.linspace(0, fit_range, 200)
                ax.semilogy(x_fit, exp_model(x_fit, *pfit), "--k", lw=1.2, alpha=0.6)
            except RuntimeError:
                pass

    ax.set_xlabel(r"Number of Iterations, $N_\mathrm{it}$", fontsize=18)
    ax.set_ylabel(r"$E(N_\mathrm{it}) - E_\infty$", fontsize=18)
    ax.set_title(label, fontsize=18, pad=8)
    ax.legend(ncol=2, loc="upper right", title=r"$N_\mathrm{layers}$",
              title_fontsize=16, fontsize=15, labelspacing=0.3, handletextpad=0.3)
    ax.grid(True, alpha=0.3, which="both")
    plt.tight_layout()
    plt.savefig(fname, bbox_inches="tight", dpi=300)
    plt.show()
    print(f"Saved {fname}")

